# Práctica 2: Function Calling, Razonamiento y Herramientas Inteligentes con gpt-5-mini

Este notebook demuestra:
1. **Function Calling** - Cómo el modelo llama funciones personalizadas
2. **Manejo de Herramientas** - Definir y procesar llamadas a funciones
3. **Razonamiento Avanzado** - Niveles de razonamiento: low, medium, high
4. **Aplicaciones Prácticas** - Ejemplos del mundo real

## Requisitos previos:
- gpt-5-mini desplegado en Azure Foundry
- API Key configurada en .env
- Variables de entorno: AZURE_OPENAI_ENDPOINT, DEPLOYMENT_NAME, AZURE_OPENAI_API_KEY

## 1. Configuración e Instalación

In [13]:
# Instalar dependencias
import subprocess
import sys

packages = ['openai>=1.42.0']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✓ Dependencias instaladas')

✓ Dependencias instaladas


In [14]:
# Cargar variables de entorno
from dotenv import load_dotenv
import os

load_dotenv()

print("✓ Variables de entorno cargadas desde .env")
print(f"  Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT', 'NO CONFIGURADO')[:50]}...")
print(f"  Deployment: {os.getenv('DEPLOYMENT_NAME', 'NO CONFIGURADO')}")

✓ Variables de entorno cargadas desde .env
  Endpoint: https://foundryjnb.openai.azure.com/openai/v1...
  Deployment: gpt-5-mini


In [15]:
# Imports
import json
from typing import Any
from openai import OpenAI

# Configuración desde .env
ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-5-mini")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

if not ENDPOINT or not API_KEY:
    raise ValueError("AZURE_OPENAI_ENDPOINT y AZURE_OPENAI_API_KEY no configurados en .env")

# Crear cliente
client = OpenAI(
    base_url=ENDPOINT,
    api_key=API_KEY
)

print(f"✓ Cliente OpenAI configurado")
print(f"  Modelo: {DEPLOYMENT_NAME}")

✓ Cliente OpenAI configurado
  Modelo: gpt-5-mini


## 2. Part 1: Function Calling básico

Ejemplo: Herramientas para cálculos matemáticos

In [16]:
# Definir herramientas (tools)
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_factorial",
            "description": "Calcula el factorial de un número (n!)",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {
                        "type": "integer",
                        "description": "El número para el cual calcular el factorial"
                    }
                },
                "required": ["n"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_fibonacci",
            "description": "Calcula el n-ésimo número de Fibonacci",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {
                        "type": "integer",
                        "description": "La posición en la secuencia de Fibonacci"
                    }
                },
                "required": ["n"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "is_prime",
            "description": "Verifica si un número es primo",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {
                        "type": "integer",
                        "description": "El número a verificar"
                    }
                },
                "required": ["n"]
            }
        }
    }
]

print("✓ 3 herramientas matemáticas definidas")

✓ 3 herramientas matemáticas definidas


In [17]:
# Implementar las funciones reales

def calculate_factorial(n: int) -> dict:
    """Calcula n!"""
    if n < 0:
        return {"error": "Factorial no definido para números negativos"}
    if n == 0 or n == 1:
        return {"result": 1, "n": n}
    
    result = 1
    for i in range(2, n + 1):
        result *= i
    return {"result": result, "n": n}

def calculate_fibonacci(n: int) -> dict:
    """Calcula el n-ésimo número de Fibonacci"""
    if n < 0:
        return {"error": "Índice no válido"}
    if n == 0:
        return {"result": 0, "n": n}
    if n == 1:
        return {"result": 1, "n": n}
    
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return {"result": b, "n": n}

def is_prime(n: int) -> dict:
    """Verifica si un número es primo"""
    if n < 2:
        return {"result": False, "n": n, "reason": "Números menores a 2 no son primos"}
    if n == 2:
        return {"result": True, "n": n}
    if n % 2 == 0:
        return {"result": False, "n": n, "reason": "Número par"}
    
    for i in range(3, int(n**0.5) + 1, 2):
        if n % i == 0:
            return {"result": False, "n": n, "reason": f"Divisible por {i}"}
    
    return {"result": True, "n": n}

# Mapeo de funciones
function_map = {
    "calculate_factorial": calculate_factorial,
    "calculate_fibonacci": calculate_fibonacci,
    "is_prime": is_prime
}

print("✓ Todas las funciones implementadas")

✓ Todas las funciones implementadas


In [18]:
def call_model_with_tools(user_message: str, tools: list) -> dict:
    """Llama al modelo y maneja function calling.
    
    Args:
        user_message: Mensaje del usuario
        tools: Lista de herramientas disponibles
    
    Returns:
        Dict con respuesta final y llamadas a funciones realizadas
    """
    
    messages = [{"role": "user", "content": user_message}]
    function_calls = []
    
    # Bucle de interacción con function calling
    for iteration in range(5):  # Máximo 5 iteraciones
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            reasoning_effort="medium"  # low, medium, high
        )
        
        # Verificar si el modelo quiere llamar una función
        if response.choices[0].finish_reason == "tool_calls":
            # Procesar las llamadas a funciones
            for tool_call in response.choices[0].message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                # Ejecutar la función
                if function_name in function_map:
                    function_result = function_map[function_name](**function_args)
                    
                    # Guardar información de la llamada
                    function_calls.append({
                        "function": function_name,
                        "arguments": function_args,
                        "result": function_result
                    })
                    
                    # Añadir resultado a los mensajes
                    messages.append({"role": "assistant", "content": response.choices[0].message.content or ""})
                    messages.append({
                        "role": "user",
                        "content": f"Función {function_name} con argumentos {function_args} retornó: {json.dumps(function_result)}"
                    })
        else:
            # El modelo ha terminado
            return {
                "response": response.choices[0].message.content,
                "function_calls": function_calls,
                "iterations": iteration + 1
            }
    
    return {
        "response": "Máximo número de iteraciones alcanzado",
        "function_calls": function_calls,
        "iterations": 5
    }

print("✓ Función call_model_with_tools definida")

✓ Función call_model_with_tools definida


In [19]:
# TEST 1: Pregunta simple con cálculo
print("\n" + "═" * 70)
print("TEST 1: Function Calling - Cálculos Varios")
print("═" * 70)

question1 = "¿Cuál es el factorial de 5? ¿Y el 10mo número de Fibonacci?"

print(f"\nPregunta: {question1}\n")
print("Procesando...\n")

result1 = call_model_with_tools(question1, tools)

print(f"Respuesta del modelo:\n{result1['response']}")
print(f"\nFunciones llamadas: {len(result1['function_calls'])}")

for i, call in enumerate(result1['function_calls'], 1):
    print(f"  {i}. {call['function']}({json.dumps(call['arguments'])}) = {call['result']['result']}")


══════════════════════════════════════════════════════════════════════
TEST 1: Function Calling - Cálculos Varios
══════════════════════════════════════════════════════════════════════

Pregunta: ¿Cuál es el factorial de 5? ¿Y el 10mo número de Fibonacci?

Procesando...

Respuesta del modelo:
El factorial de 5 es 120 (5! = 5×4×3×2×1 = 120).  
El 10.º número de Fibonacci es 55.

Funciones llamadas: 2
  1. calculate_factorial({"n": 5}) = 120
  2. calculate_fibonacci({"n": 10}) = 55


In [20]:
# TEST 2: Verificación de números primos
print("\n" + "═" * 70)
print("TEST 2: Function Calling - Análisis de Números Primos")
print("═" * 70)

question2 = "¿Cuáles son primos entre estos: 17, 21, 29, 33, 41?"

print(f"\nPregunta: {question2}\n")
print("Procesando...\n")

result2 = call_model_with_tools(question2, tools)

print(f"Respuesta del modelo:\n{result2['response']}")
print(f"\nFunciones llamadas: {len(result2['function_calls'])}")

for i, call in enumerate(result2['function_calls'], 1):
    print(f"  {i}. {call['function']}({call['arguments']['n']}) = {call['result']['result']}")


══════════════════════════════════════════════════════════════════════
TEST 2: Function Calling - Análisis de Números Primos
══════════════════════════════════════════════════════════════════════

Pregunta: ¿Cuáles son primos entre estos: 17, 21, 29, 33, 41?

Procesando...

Respuesta del modelo:
Los primos son 17, 29 y 41.  
No son primos: 21 (3×7) y 33 (3×11).

Funciones llamadas: 5
  1. is_prime(17) = True
  2. is_prime(21) = False
  3. is_prime(29) = True
  4. is_prime(33) = False
  5. is_prime(41) = True


## 3. Part 2: Function Calling Avanzado

Ejemplo: Sistema de información sobre películas

In [21]:
# Base de datos simulada de películas
movie_database = {
    "1": {"title": "Inception", "director": "Christopher Nolan", "year": 2010, "rating": 8.8, "genre": ["Sci-Fi", "Thriller"]},
    "2": {"title": "The Matrix", "director": "Wachowski", "year": 1999, "rating": 8.7, "genre": ["Sci-Fi", "Action"]},
    "3": {"title": "Parasite", "director": "Bong Joon-ho", "year": 2019, "rating": 8.6, "genre": ["Drama", "Thriller"]},
    "4": {"title": "Pulp Fiction", "director": "Quentin Tarantino", "year": 1994, "rating": 8.9, "genre": ["Crime", "Drama"]},
    "5": {"title": "Interstellar", "director": "Christopher Nolan", "year": 2014, "rating": 8.6, "genre": ["Sci-Fi", "Drama"]}
}

# Herramientas para películas
movie_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_movies_by_director",
            "description": "Busca películas por director",
            "parameters": {
                "type": "object",
                "properties": {
                    "director_name": {
                        "type": "string",
                        "description": "Nombre del director"
                    }
                },
                "required": ["director_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_movies_by_genre",
            "description": "Busca películas por género",
            "parameters": {
                "type": "object",
                "properties": {
                    "genre": {
                        "type": "string",
                        "description": "El género (Sci-Fi, Drama, Action, Crime, Thriller)"
                    }
                },
                "required": ["genre"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Obtiene detalles de una película por ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "string",
                        "description": "El ID de la película (1-5)"
                    }
                },
                "required": ["movie_id"]
            }
        }
    }
]

# Actualizar mapeo de funciones
def search_movies_by_director(director_name: str) -> dict:
    results = []
    for movie_id, movie in movie_database.items():
        if director_name.lower() in movie["director"].lower():
            results.append({"id": movie_id, "title": movie["title"], "year": movie["year"], "rating": movie["rating"]})
    return {"director": director_name, "results": results, "found": len(results)}

def search_movies_by_genre(genre: str) -> dict:
    results = []
    for movie_id, movie in movie_database.items():
        if genre in movie["genre"]:
            results.append({"id": movie_id, "title": movie["title"], "year": movie["year"], "rating": movie["rating"]})
    return {"genre": genre, "results": results, "found": len(results)}

def get_movie_details(movie_id: str) -> dict:
    if movie_id in movie_database:
        return {"found": True, "movie": movie_database[movie_id]}
    return {"found": False, "error": "Película no encontrada"}

# Actualizar mapeo
function_map.update({
    "search_movies_by_director": search_movies_by_director,
    "search_movies_by_genre": search_movies_by_genre,
    "get_movie_details": get_movie_details
})

print("✓ Base de datos de películas y herramientas definidas")

✓ Base de datos de películas y herramientas definidas


In [22]:
# TEST 3: Búsqueda de películas
print("\n" + "═" * 70)
print("TEST 3: Function Calling - Sistema de Películas")
print("═" * 70)

question3 = "¿Qué películas de ciencia ficción hay? Dame detalles de la mejor valorada."

print(f"\nPregunta: {question3}\n")
print("Procesando...\n")

result3 = call_model_with_tools(question3, movie_tools)

print(f"Respuesta del modelo:\n{result3['response']}")
print(f"\nFunciones llamadas: {len(result3['function_calls'])}")

for i, call in enumerate(result3['function_calls'], 1):
    args_str = json.dumps(call['arguments'])
    print(f"  {i}. {call['function']}({args_str})")


══════════════════════════════════════════════════════════════════════
TEST 3: Function Calling - Sistema de Películas
══════════════════════════════════════════════════════════════════════

Pregunta: ¿Qué películas de ciencia ficción hay? Dame detalles de la mejor valorada.

Procesando...

Respuesta del modelo:
He encontrado 3 películas de ciencia ficción:

- Inception (2010) — valoración: 8.8  
- The Matrix (1999) — valoración: 8.7  
- Interstellar (2014) — valoración: 8.6

La mejor valorada es Inception. Detalles disponibles:
- Título: Inception  
- Director: Christopher Nolan  
- Año: 2010  
- Valoración: 8.8  
- Géneros: Sci-Fi, Thriller

¿Quieres que obtenga más detalles (sinopsis, reparto, duración) de Inception o que traiga los detalles de The Matrix o Interstellar?

Funciones llamadas: 2
  1. search_movies_by_genre({"genre": "Sci-Fi"})
  2. get_movie_details({"movie_id": "1"})


In [23]:
# TEST 4: Búsqueda por director
print("\n" + "═" * 70)
print("TEST 4: Function Calling - Películas de un Director")
print("═" * 70)

question4 = "¿Cuáles son todas las películas de Christopher Nolan que tenemos? Compara sus valoraciones."

print(f"\nPregunta: {question4}\n")
print("Procesando...\n")

result4 = call_model_with_tools(question4, movie_tools)

print(f"Respuesta del modelo:\n{result4['response']}")
print(f"\nFunciones llamadas: {len(result4['function_calls'])}")

for i, call in enumerate(result4['function_calls'], 1):
    args_str = json.dumps(call['arguments'])
    print(f"  {i}. {call['function']}({args_str})")


══════════════════════════════════════════════════════════════════════
TEST 4: Function Calling - Películas de un Director
══════════════════════════════════════════════════════════════════════

Pregunta: ¿Cuáles son todas las películas de Christopher Nolan que tenemos? Compara sus valoraciones.

Procesando...

Respuesta del modelo:
Tenemos 2 películas de Christopher Nolan en la base de datos:

- Inception (2010) — valoración: 8.8  
- Interstellar (2014) — valoración: 8.6

Comparación rápida:
- Inception tiene la valoración más alta: 8.8 vs 8.6 de Interstellar.  
- Diferencia: 0.2 puntos (Inception > Interstellar).  
- Ambas están muy bien valoradas y la diferencia es pequeña.

¿Quieres que obtenga más detalles de alguna (sinopsis, reparto, duración)? Puedo traer la ficha completa de cualquiera usando sus IDs (1 para Inception, 5 para Interstellar).

Funciones llamadas: 1
  1. search_movies_by_director({"director_name": "Christopher Nolan"})


## 4. Resumen: Function Calling

### ¿Qué es Function Calling?
Function Calling permite que el modelo OpenAI:
1. **Identifique** cuándo necesita información externa
2. **Seleccione** la función apropiada para usar
3. **Genere** los argumentos correctos automáticamente
4. **Integre** los resultados en su respuesta final

### Casos de Uso
- **APIs Externas**: Obtener datos de APIs (meteorología, cotizaciones, etc.)
- **Bases de Datos**: Consultar información estructurada
- **Cálculos**: Realizar operaciones matemáticas precisas
- **Búsquedas**: Consultar índices o catálogos
- **Integraciones**: Conectar con sistemas externos

### Ventajas
✓ El modelo elige automáticamente qué función usar
✓ Respuestas más precisas con datos actualizados
✓ Integración transparente con sistemas existentes
✓ Flujo conversacional natural
✓ Validación de argumentos automática

In [24]:
print("\n" + "═" * 70)
print("CONCLUSIÓN")
print("═" * 70)
print("✓ Function Calling demostrado con múltiples ejemplos")
print("✓ Herramientas personalizadas funcionando correctamente")
print("✓ El modelo gpt-5-mini puede:")
print("  - Identificar cuándo necesita herramientas")
print("  - Seleccionar la función correcta")
print("  - Generar argumentos automáticamente")
print("  - Integrar resultados en respuestas coherentes")
print("\n¡Listo para usar en aplicaciones reales!")
print("═" * 70)


══════════════════════════════════════════════════════════════════════
CONCLUSIÓN
══════════════════════════════════════════════════════════════════════
✓ Function Calling demostrado con múltiples ejemplos
✓ Herramientas personalizadas funcionando correctamente
✓ El modelo gpt-5-mini puede:
  - Identificar cuándo necesita herramientas
  - Seleccionar la función correcta
  - Generar argumentos automáticamente
  - Integrar resultados en respuestas coherentes

¡Listo para usar en aplicaciones reales!
══════════════════════════════════════════════════════════════════════


## 5. Razonamiento Avanzado: Comparación de Niveles (low, medium, high)

Cuando uses un modelo con capacidad de razonamiento, puedes experimentar con diferentes niveles de esfuerzo cognitivo.


In [25]:
def call_model_with_high_reasoning(user_message: str, tools: list, reasoning_level: str = "high") -> dict:
    """Llama al modelo con nivel de razonamiento especificado.
    
    Args:
        user_message: Mensaje del usuario
        tools: Lista de herramientas disponibles
        reasoning_level: "low", "medium", o "high"
    
    Returns:
        Dict con respuesta final, llamadas a funciones e información de razonamiento
    """
    
    messages = [{"role": "user", "content": user_message}]
    function_calls = []
    
    # Bucle de interacción con function calling
    for iteration in range(5):  # Máximo 5 iteraciones
        try:
            response = client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=messages,
                tools=tools,
                tool_choice="auto",
                reasoning_effort=reasoning_level  # Usar el nivel especificado
            )
        except Exception as e:
            # Si reasoning_effort no es soportado, usar sin él
            if "reasoning_effort" in str(e):
                print(f"⚠️ Nota: {DEPLOYMENT_NAME} no soporta reasoning_effort, usando sin este parámetro")
                response = client.chat.completions.create(
                    model=DEPLOYMENT_NAME,
                    messages=messages,
                    tools=tools,
                    tool_choice="auto"
                )
            else:
                raise
        
        # Verificar si el modelo quiere llamar una función
        if response.choices[0].finish_reason == "tool_calls":
            # Procesar las llamadas a funciones
            for tool_call in response.choices[0].message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                # Ejecutar la función
                if function_name in function_map:
                    function_result = function_map[function_name](**function_args)
                    
                    # Guardar información de la llamada
                    function_calls.append({
                        "function": function_name,
                        "arguments": function_args,
                        "result": function_result
                    })
                    
                    # Añadir resultado a los mensajes
                    messages.append({"role": "assistant", "content": response.choices[0].message.content or ""})
                    messages.append({
                        "role": "user",
                        "content": f"Función {function_name} con argumentos {function_args} retornó: {json.dumps(function_result)}"
                    })
        else:
            # El modelo ha terminado
            return {
                "response": response.choices[0].message.content,
                "function_calls": function_calls,
                "iterations": iteration + 1,
                "reasoning_level": reasoning_level
            }
    
    return {
        "response": "Máximo número de iteraciones alcanzado",
        "function_calls": function_calls,
        "iterations": 5,
        "reasoning_level": reasoning_level
    }

print("✓ Función call_model_with_high_reasoning definida")

✓ Función call_model_with_high_reasoning definida


In [26]:
# TEST: Comparación entre razonamiento Medium vs High
print("\n" + "═" * 70)
print("TEST: Comparación de Niveles de Razonamiento (Medium vs High)")
print("═" * 70)

question_comparison = "¿Cuál es el factorial de 7? Análiza si 97 es un número primo complejo."

print(f"\nPregunta: {question_comparison}\n")

# Ejecutar con nivel MEDIUM
print("▶ Ejecutando con reasoning_effort='medium'...\n")
result_medium = call_model_with_high_reasoning(question_comparison, tools, reasoning_level="medium")

print(f"Response (Medium):\n{result_medium['response']}")
print(f"Funciones llamadas: {len(result_medium['function_calls'])}")
for i, call in enumerate(result_medium['function_calls'], 1):
    print(f"  {i}. {call['function']}({json.dumps(call['arguments'])}) = {call['result'].get('result', call['result'])}")

# Ejecutar con nivel HIGH
print("\n" + "-" * 70)
print("▶ Ejecutando con reasoning_effort='high'...\n")
result_high = call_model_with_high_reasoning(question_comparison, tools, reasoning_level="high")

print(f"Response (High):\n{result_high['response']}")
print(f"Funciones llamadas: {len(result_high['function_calls'])}")
for i, call in enumerate(result_high['function_calls'], 1):
    print(f"  {i}. {call['function']}({json.dumps(call['arguments'])}) = {call['result'].get('result', call['result'])}")

# Comparación
print("\n" + "-" * 70)
print("COMPARACIÓN:")
print(f"Medium: {len(result_medium['function_calls'])} funciones, {result_medium['iterations']} iteraciones")
print(f"High:   {len(result_high['function_calls'])} funciones, {result_high['iterations']} iteraciones")
print("\nNota: Con 'high' el modelo dedica más tiempo a razonamiento, lo que puede")
print("resultar en respuestas más precisas pero más lentas.")



══════════════════════════════════════════════════════════════════════
TEST: Comparación de Niveles de Razonamiento (Medium vs High)
══════════════════════════════════════════════════════════════════════

Pregunta: ¿Cuál es el factorial de 7? Análiza si 97 es un número primo complejo.

▶ Ejecutando con reasoning_effort='medium'...

Response (Medium):
7! = 5040.

Sobre si 97 es un “número primo complejo”:

- Si por “primo complejo” te refieres a un elemento primo en el campo de los números complejos C: eso no tiene sentido útil, porque C es un cuerpo (campo) y en un campo no hay elementos primos distintos de cero (todos los elementos no nulos son invertibles).

- Si te refieres a un primo en los enteros de Gauss Z[i] (es decir, un “Gaussian prime”): los primos racionales p se comportan así en Z[i]:
  - p = 2 es ramificado,
  - p ≡ 1 (mod 4) se descompone (no es primo en Z[i]),
  - p ≡ 3 (mod 4) permanece primo en Z[i].

  Como 97 ≡ 1 (mod 4) y 97 = 9^2 + 4^2, se factoriza en Z[i]:
  97